# Dengue forecasting — driver notebook

Pathway B. The model and harness code lives in `src/forecasting/`; this notebook is a thin
driver that loads the data, runs every model through the rolling-origin simulator, and
prints the headline + per-district tables that feed `docs/EVAL_DESIGN.md`.

Sections:
1. Setup
2. Load data and build the panel
3. EDA — case curves and seasonality
4. Harness verification — does our `SeasonalNaive` match the shipped baseline?
5. Run every model through the simulator
6. Headline metrics
7. Per-district and surge-season slices


## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Make src/ importable from the notebook
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from forecasting import (
    load_data, build_panel,
    LastWeekNaive, SeasonalNaive, PoissonForecaster,
    SeasonalNaivePlus, SeasonalNaivePlusV2,
    XGBoostForecaster, XGBoostForecasterV2,
    run_simulation, compute_metrics, compute_metrics_grouped,
)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
pd.set_option("display.max_columns", None)


## 2. Load data and build the panel

In [ ]:
cases, weather, districts, baseline = load_data()
df = build_panel(cases, weather, districts)

print(f"Panel: {df.shape}")
print(f"Districts: {sorted(df['district_id'].unique())}")
print(f"Date range: {df['week_start'].min().date()} to {df['week_start'].max().date()}")
print(f"Missing values:\n{df.isna().sum()}")
df.head(3)


## 3. EDA — case curves and seasonality

In [ ]:
fig, axes = plt.subplots(6, 1, figsize=(14, 18), sharex=True)
for ax, did in zip(axes, ['D01', 'D02', 'D03', 'D04', 'D05', 'D06']):
    sub = df[df['district_id'] == did]
    name = sub['canonical_name'].iloc[0]
    tier = sub['tier'].iloc[0]
    pop_lakhs = sub['population'].iloc[0] / 100000
    ax.plot(sub['week_start'], sub['cases'], linewidth=1.2)
    ax.set_title(f"{did} — {name}  (Tier {tier}, pop {pop_lakhs:.0f} lakh)", loc='left', fontsize=11)
    ax.set_ylabel("Cases / week")
    ax.axvline(pd.Timestamp("2024-01-01"), color='red', linestyle='--', alpha=0.4, label='2023 → 2024')
axes[0].legend(loc='upper right')
axes[-1].set_xlabel("Week")
plt.suptitle("Weekly dengue cases by district (2023-2024)", fontsize=14, y=1.00)
plt.tight_layout()
plt.show()


In [ ]:
# Seasonality: total cases per week-of-year, 2023 vs 2024
df['year'] = df['week_start'].dt.isocalendar().year
df['week_of_year'] = df['week_start'].dt.isocalendar().week
totals = df.groupby(['year', 'week_of_year'])['cases'].sum().reset_index()

fig, ax = plt.subplots(figsize=(13, 5))
for yr in sorted(totals['year'].unique()):
    sub = totals[totals['year'] == yr]
    ax.plot(sub['week_of_year'], sub['cases'], marker='o', markersize=3, linewidth=1.5, label=str(yr))
ax.set_xlabel("Week of year (ISO)")
ax.set_ylabel("Total cases across all 6 districts")
ax.set_title("Seasonality: total weekly cases, 2023 vs 2024")
ax.axvspan(22, 36, alpha=0.1, color='orange', label='Surge season (W22–W36)')
ax.legend()
plt.tight_layout()
plt.show()


## 4. Harness verification

Before any modelling: does our `SeasonalNaive` reproduce the shipped baseline file row-for-row?
A match on the well-formed rows means the harness can be trusted to score new models fairly.

In [ ]:
sn = SeasonalNaive()
sn.fit(df)

check = baseline.copy()
check['issue_dt']  = pd.to_datetime(check['issue_iso_week']  + '-1', format='%G-W%V-%u')
check['target_dt'] = pd.to_datetime(check['target_iso_week'] + '-1', format='%G-W%V-%u')
check['real_horizon'] = (check['target_dt'] - check['issue_dt']).dt.days // 7
check['well_formed']  = check['real_horizon'] == check['horizon_weeks']

check['our_mean'] = check.apply(
    lambda r: sn.predict(r['issue_dt'], int(r['horizon_weeks']), r['district_id'])['mean'],
    axis=1,
)
check['matches'] = (check['our_mean'] - check['mean']).abs() < 0.5

print(f"Total rows: {len(check)}")
print(f"Well-formed rows:                       {check['well_formed'].sum()}")
print(f"Well-formed AND matching ours:          {(check['well_formed'] & check['matches']).sum()}")
print(f"Well-formed but NOT matching (concern): {(check['well_formed'] & ~check['matches']).sum()}")


## 5. Run every model through the simulator

48 issue-weeks (2024-W01 to 2024-W48) × 6 districts × 2 horizons = 576 forecasts per model.

In [ ]:
first_issue = pd.to_datetime("2024-W01-1", format='%G-W%V-%u')
last_issue  = pd.to_datetime("2024-W48-1", format='%G-W%V-%u')

tier_map = districts.set_index('district_id')['tier'].to_dict()

forecasts = {}
for model_cls in [LastWeekNaive, SeasonalNaive, PoissonForecaster,
                  SeasonalNaivePlus, SeasonalNaivePlusV2,
                  XGBoostForecaster, XGBoostForecasterV2]:
    m = model_cls()
    print(f"Running {m.name} ...")
    fc = run_simulation(m, df, first_issue, last_issue, refit_each_step=False)
    fc['tier'] = fc['district_id'].map(tier_map)
    fc['target_week_of_year'] = fc['target_week_dt'].dt.isocalendar().week
    forecasts[m.name] = fc

print({name: len(fc) for name, fc in forecasts.items()})


## 6. Headline metrics

In [ ]:
headline = pd.concat(
    [compute_metrics(fc, name) for name, fc in forecasts.items()],
    ignore_index=True,
)
print(headline.to_string(index=False))


## 7. Per-district and surge-season slices

The per-district view is the main finding: no single model wins everywhere.
See `docs/EVAL_DESIGN.md` §7 for the per-district assignment recommendation.

In [ ]:
def surge_only(fc):
    return fc[(fc['target_week_of_year'] >= 22) & (fc['target_week_of_year'] <= 36)]

for name in ('seasonal_naive', 'xgboost_with_sn'):
    fc = forecasts[name]
    print(f"\n=== {name} — surge-season per-district ===")
    s = compute_metrics_grouped(surge_only(fc), name, ['district_id'])
    print(s[['district_id', 'n', 'mean_actual', 'MAE', 'MAE_pct_of_mean', 'bias', 'coverage_80']].to_string(index=False))


In [ ]:
# Side-by-side: SN vs XGB-with-SN, surge season, per district
sn_surge  = compute_metrics_grouped(surge_only(forecasts['seasonal_naive']),  'sn',  ['district_id'])
xgb_surge = compute_metrics_grouped(surge_only(forecasts['xgboost_with_sn']), 'xgb', ['district_id'])

compare = sn_surge[['district_id', 'mean_actual', 'MAE_pct_of_mean']].rename(
    columns={'MAE_pct_of_mean': 'SN_MAE%'}
).merge(
    xgb_surge[['district_id', 'MAE_pct_of_mean']].rename(columns={'MAE_pct_of_mean': 'XGB_MAE%'}),
    on='district_id',
)
print(compare.round(1).to_string(index=False))


---

For the full reasoning behind metric choices, model decisions, failure-mode probes,
and the per-district pilot recommendation, see [`docs/EVAL_DESIGN.md`](../docs/EVAL_DESIGN.md).
